In [ ]:
import numpy as np
import pandas as pd
from config import CLEAN_DATA_PATH ,RANDOM_STATE ,BINARY_COLS ,MULTI_CAT_COLS ,NUMERIC_COLS ,REGRESSION_TARGET ,CLASSIFICATION_TARGET
from scipy.stats import randint, uniform, loguniform

# Preproccsing 
from sklearn.model_selection import train_test_split ,GridSearchCV ,RandomizedSearchCV ,KFold ,StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler , OneHotEncoder , PolynomialFeatures 

# Models
from sklearn.linear_model import LinearRegression ,Ridge ,Lasso ,LogisticRegression 
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC 
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier , GradientBoostingClassifier , RandomForestRegressor ,GradientBoostingRegressor

# Metics
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [3]:
df = pd.read_csv(CLEAN_DATA_PATH)

In [4]:
def make_preprocessor():
    return ColumnTransformer(transformers=[
        ("num", StandardScaler(), NUMERIC_COLS),
        ("bin", OneHotEncoder(drop="if_binary", handle_unknown="ignore"), BINARY_COLS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), MULTI_CAT_COLS),
    ])


In [5]:
def make_poly_preprocessor():
    numeric_pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("poly", PolynomialFeatures(include_bias=False)),
    ])
    return ColumnTransformer(transformers=[
        ("num", numeric_pipe, NUMERIC_COLS),
        ("bin", OneHotEncoder(drop="if_binary", handle_unknown="ignore"), BINARY_COLS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), MULTI_CAT_COLS),
    ])

In [6]:
regression_results = []
classification_results = []

In [7]:
def log_regression_result(results_list, model_name, y_true, y_pred, best_params=None):
    row = {
        "Model": model_name,
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
        "Best_Params": best_params if best_params is not None else "-",
    }
    results_list.append(row)
    print(f"{model_name:30s} | RMSE: {row['RMSE']:8.2f} | MAE: {row['MAE']:8.2f} | "
          f"R2: {row['R2']:.4f} | Params: {row['Best_Params']}")
    return row


# Reggression 
### Target value CLTV


In [8]:
reg_feature_cols = NUMERIC_COLS + BINARY_COLS + MULTI_CAT_COLS
X_reg = df[reg_feature_cols]
y_reg = df[REGRESSION_TARGET]
 
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=RANDOM_STATE
)

In [9]:
def train_polynomial_regression(X_train, X_test, y_train, y_test, results_list):
    pipeline = Pipeline([
        ("preprocessor", make_poly_preprocessor()),
        ("model", LinearRegression()),
    ])
    param_grid = {"preprocessor__num__poly__degree": [3, 4, 5]}
 
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="r2",
        cv=KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
        n_jobs=-1,
    )
    search.fit(X_train, y_train)
 
    best_pipeline = search.best_estimator_
    preds = best_pipeline.predict(X_test)
 
    log_regression_result(results_list, "Polynomial Regression", y_test, preds,
                           best_params=search.best_params_)
train_polynomial_regression(X_reg_train, X_reg_test, y_reg_train, y_reg_test, regression_results)

Polynomial Regression          | RMSE:  1038.58 | MAE:   883.20 | R2: 0.2060 | Params: {'preprocessor__num__poly__degree': 4}


In [10]:
def train_lasso_regression(X_train, X_test, y_train, y_test, results_list):
    pipeline = Pipeline([
        ("preprocessor", make_preprocessor()),
        ("model", Lasso(random_state=RANDOM_STATE, max_iter=10000)),
    ])
    param_grid = {"model__alpha": [5.0, 6.0, 7.0, 8.0, 9.0, 10.0]}
 
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="r2",
        cv=KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
        n_jobs=-1,
    )
    search.fit(X_train, y_train)
 
    best_pipeline = search.best_estimator_
    preds = best_pipeline.predict(X_test)
 
    log_regression_result(results_list, "Lasso Regression", y_test, preds,
                           best_params=search.best_params_)
train_lasso_regression(X_reg_train, X_reg_test, y_reg_train, y_reg_test, regression_results)



Lasso Regression               | RMSE:  1061.70 | MAE:   898.20 | R2: 0.1702 | Params: {'model__alpha': 7.0}


In [11]:
def train_ridge_regression(X_train, X_test, y_train, y_test, results_list):
    pipeline = Pipeline([
        ("preprocessor", make_preprocessor()),
        ("model", Ridge(random_state=RANDOM_STATE)),
    ])
    param_grid = {"model__alpha": [175.0, 200.0, 225.0, 250.0]}
 
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="r2",
        cv=KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
        n_jobs=-1,
    )
    search.fit(X_train, y_train)
 
    best_pipeline = search.best_estimator_
    preds = best_pipeline.predict(X_test)
 
    log_regression_result(results_list, "Ridge Regression", y_test, preds,
                           best_params=search.best_params_)
train_ridge_regression(X_reg_train, X_reg_test, y_reg_train, y_reg_test, regression_results)

Ridge Regression               | RMSE:  1062.02 | MAE:   898.43 | R2: 0.1697 | Params: {'model__alpha': 225.0}


In [12]:
def train_random_forest_regressor(X_train, X_test, y_train, y_test, results_list):

    pipeline = Pipeline([
        ("preprocessor", make_preprocessor()),
        ("model", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)),
    ])
    param_distributions = {
        "model__n_estimators": randint(300, 400),
        "model__max_depth": [1, 2, 3],
        "model__min_samples_leaf": randint(1, 6),
        "model__max_features": ["sqrt", "log2", 0.5, 1.0],
    }
 
    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_distributions,
        n_iter=30,
        scoring="r2",
        cv=KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    search.fit(X_train, y_train)
 
    best_pipeline = search.best_estimator_
    preds = best_pipeline.predict(X_test)
 
    log_regression_result(results_list, "Random Forest Regressor", y_test, preds, best_params=search.best_params_)

train_random_forest_regressor(X_reg_train, X_reg_test, y_reg_train, y_reg_test, regression_results)


Random Forest Regressor        | RMSE:  1025.55 | MAE:   872.92 | R2: 0.2258 | Params: {'model__max_depth': 1, 'model__max_features': 1.0, 'model__min_samples_leaf': 1, 'model__n_estimators': 307}


In [13]:
def train_gradient_boosting_regressor(X_train, X_test, y_train, y_test, results_list):
 
    pipeline = Pipeline([
        ("preprocessor", make_preprocessor()),
        ("model", GradientBoostingRegressor(random_state=RANDOM_STATE)),
    ])
    param_distributions = {
        "model__n_estimators": randint(300,400),
        "model__learning_rate": loguniform(0.001, 0.05),
        "model__max_depth": [1, 2, 3],
        "model__subsample": uniform(0.6, 0.4), 
    }
 
    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_distributions,
        n_iter=30,
        scoring="r2",
        cv=KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    search.fit(X_train, y_train)
 
    best_pipeline = search.best_estimator_
    preds = best_pipeline.predict(X_test)
 
    log_regression_result(results_list, "Gradient Boosting Regressor", y_test, preds, best_params=search.best_params_)

train_gradient_boosting_regressor(X_reg_train, X_reg_test, y_reg_train, y_reg_test, regression_results)

Gradient Boosting Regressor    | RMSE:  1026.00 | MAE:   873.24 | R2: 0.2251 | Params: {'model__learning_rate': np.float64(0.008263686276667369), 'model__max_depth': 1, 'model__n_estimators': 335, 'model__subsample': np.float64(0.9771414282231924)}


# Regression Results


In [14]:
reg_df = pd.DataFrame(regression_results)
reg_df = reg_df.sort_values(by = 'R2',ascending=False)
reg_df.to_csv('Hyper results/reg_results_hyper.csv',index= False)
reg_df

,Model,RMSE,MAE,R2,Best_Params
3,Random Forest Regressor,1025.550316,872.918115,0.225773,"{'model__max_depth': 1, 'model__max_features':..."
4,Gradient Boosting Regressor,1026.002477,873.243050,0.225090,"{'model__learning_rate': 0.008263686276667369,..."
0,Polynomial Regression,1038.578161,883.200988,0.205978,{'preprocessor__num__poly__degree': 4}
1,Lasso Regression,1061.702289,898.197690,0.170226,{'model__alpha': 7.0}
2,Ridge Regression,1062.019030,898.426632,0.169731,{'model__alpha': 225.0}


# Classification 
### Target value Churn


In [15]:
clf_feature_cols = NUMERIC_COLS + BINARY_COLS + MULTI_CAT_COLS + [REGRESSION_TARGET]
X_clf = df[clf_feature_cols]
y_clf = df[CLASSIFICATION_TARGET]
 
X_clf_train, X_clf_test, y_clf_train, y_clf_test = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=RANDOM_STATE, stratify=y_clf
)


In [16]:
def log_classification_result(results_list, model_name, y_true, y_pred, y_proba,best_params=None):
    row = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
        "ROC_AUC": roc_auc_score(y_true, y_proba),
        "Best_Params": best_params if best_params is not None else "-",
        }
    
    results_list.append(row)
    print(f"{model_name:20s} | Acc: {row['Accuracy']:.4f} | Prec: {row['Precision']:.4f} | "
          f"Rec: {row['Recall']:.4f} | F1: {row['F1']:.4f} | AUC: {row['ROC_AUC']:.4f} | "
          f"Params: {row['Best_Params']}")
    return row


In [17]:
def train_logistic_regression(X_train, X_test, y_train, y_test, results_list):

    pipeline = Pipeline([
        ("preprocessor", make_preprocessor()),
        ("model", LogisticRegression(class_weight="balanced" ,solver="liblinear" ,max_iter=1000 ,random_state=RANDOM_STATE)),
    ])
    param_grid = {
        "model__C": [10.0,12.5, 14,15.0,17.5,20],
    }
 
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="f1",
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
        n_jobs=-1,
    )
    search.fit(X_train, y_train)
 
    best_pipeline = search.best_estimator_
    preds = best_pipeline.predict(X_test)
    probs = best_pipeline.predict_proba(X_test)[:, 1]
 
    log_classification_result(results_list, "Logistic Regression", y_test, preds, probs, best_params=search.best_params_)
train_logistic_regression(X_clf_train, X_clf_test, y_clf_train, y_clf_test, classification_results)


Logistic Regression  | Acc: 0.7438 | Prec: 0.5113 | Rec: 0.7834 | F1: 0.6188 | AUC: 0.8481 | Params: {'model__C': 12.5}


In [18]:
def train_knn(X_train, X_test, y_train, y_test, results_list):
    pipeline = Pipeline([
        ("preprocessor", make_preprocessor()),
        ("model", KNeighborsClassifier()),
    ])
    param_grid = {
        "model__n_neighbors": [5, 11, 21, 31],
        "model__weights": ["uniform", "distance"],
        "model__metric": ["euclidean", "manhattan"],
    }
 
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="f1",
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
        n_jobs=-1,
    )
    search.fit(X_train, y_train)
 
    best_pipeline = search.best_estimator_
    preds = best_pipeline.predict(X_test)
    probs = best_pipeline.predict_proba(X_test)[:, 1]
 
    log_classification_result(results_list, "KNN", y_test, preds, probs, best_params=search.best_params_)
train_knn(X_clf_train, X_clf_test, y_clf_train, y_clf_test, classification_results)



KNN                  | Acc: 0.7779 | Prec: 0.5818 | Rec: 0.5802 | F1: 0.5810 | AUC: 0.8267 | Params: {'model__metric': 'manhattan', 'model__n_neighbors': 21, 'model__weights': 'uniform'}


In [41]:
def train_svm(X_train, X_test, y_train, y_test, results_list):
    pipeline = Pipeline([
        ("preprocessor", make_preprocessor()),
        ("model", SVC(probability=True, class_weight="balanced", random_state=RANDOM_STATE)),
    ])
    param_grid = {
        "model__C": [0.1, 1.0, 10.0],
        "model__kernel": ["linear", "rbf"],
        "model__gamma": ["scale", 0.01],
    }
 
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="f1",
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
        n_jobs=-1,
    )
    search.fit(X_train, y_train)
 
    best_pipeline = search.best_estimator_
    preds = best_pipeline.predict(X_test)
    probs = best_pipeline.predict_proba(X_test)[:, 1]
 
    log_classification_result(results_list, "SVM", y_test, preds, probs, best_params=search.best_params_)
train_svm(X_clf_train, X_clf_test, y_clf_train, y_clf_test, classification_results)


SVM                  | Acc: 0.7573 | Prec: 0.5287 | Rec: 0.7888 | F1: 0.6330 | AUC: 0.8367 | Params: {'model__C': 1.0, 'model__gamma': 'scale', 'model__kernel': 'rbf'}


In [39]:
def train_decision_tree(X_train, X_test, y_train, y_test, results_list):
    pipeline = Pipeline([
        ("preprocessor", make_preprocessor()),
        ("model", DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE)),
    ])
    param_grid = {
        "model__max_depth": [3, 4, 5, 6, 7],
        "model__min_samples_leaf": [1, 3, 5, 6, 7, 8, 9, 10, 12, 15],
    }
 
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="f1",
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
        n_jobs=-1,
    )
    search.fit(X_train, y_train)
 
    best_pipeline = search.best_estimator_
    preds = best_pipeline.predict(X_test)
    probs = best_pipeline.predict_proba(X_test)[:, 1]
 
    log_classification_result(results_list, "Decision Tree", y_test, preds, probs, best_params=search.best_params_)
train_decision_tree(X_clf_train, X_clf_test, y_clf_train, y_clf_test, classification_results)



Decision Tree        | Acc: 0.7466 | Prec: 0.5150 | Rec: 0.7781 | F1: 0.6198 | AUC: 0.8359 | Params: {'model__max_depth': 6, 'model__min_samples_leaf': 8}


In [33]:
def train_random_forest(X_train, X_test, y_train, y_test, results_list):
    pipeline = Pipeline([
        ("preprocessor", make_preprocessor()),
        ("model", RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)),
    ])
    param_distributions = {
        "model__n_estimators": randint(300, 500),
        "model__max_depth": [3, 5, 8, 9, 10, 11, 12, 13, 14, 15, 16],
        "model__min_samples_leaf": randint(1, 30),
        "model__max_features": ["sqrt", "log2", 0.5, 1.0],
    }
 
    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_distributions,
        n_iter=30,
        scoring="f1",
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    search.fit(X_train, y_train)
 
    best_pipeline = search.best_estimator_
    preds = best_pipeline.predict(X_test)
    probs = best_pipeline.predict_proba(X_test)[:, 1]
 
    log_classification_result(results_list, "Random Forest", y_test, preds, probs, best_params=search.best_params_)

train_random_forest(X_clf_train, X_clf_test, y_clf_train, y_clf_test, classification_results)



Random Forest        | Acc: 0.7637 | Prec: 0.5380 | Rec: 0.7754 | F1: 0.6353 | AUC: 0.8538 | Params: {'model__max_depth': 13, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 12, 'model__n_estimators': 435}


In [43]:
def train_gradient_boosting(X_train, X_test, y_train, y_test, results_list):
    pipeline = Pipeline([
        ("preprocessor", make_preprocessor()),
        ("model", GradientBoostingClassifier(random_state=RANDOM_STATE)),
    ])
    param_distributions = {
        "model__n_estimators": randint(50, 200),
        "model__learning_rate": loguniform(0.001, 0.3),
        "model__max_depth": randint(2, 6),
        "model__subsample": uniform(0.6, 0.4),
    }
 
    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_distributions,
        n_iter=30,
        scoring="f1",
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    search.fit(X_train, y_train)
 
    best_pipeline = search.best_estimator_
    preds = best_pipeline.predict(X_test)
    probs = best_pipeline.predict_proba(X_test)[:, 1]
 
    log_classification_result(results_list, "Gradient Boosting", y_test, preds, probs, best_params=search.best_params_)

    
train_gradient_boosting(X_clf_train, X_clf_test, y_clf_train, y_clf_test, classification_results)


ValueError: Invalid parameter 'scale_pos_weight' for estimator GradientBoostingClassifier(learning_rate=np.float64(0.03009332754366945),
                           n_estimators=124, random_state=42). Valid parameters are: ['ccp_alpha', 'criterion', 'init', 'learning_rate', 'loss', 'max_depth', 'max_features', 'max_leaf_nodes', 'min_impurity_decrease', 'min_samples_leaf', 'min_samples_split', 'min_weight_fraction_leaf', 'n_estimators', 'n_iter_no_change', 'random_state', 'subsample', 'tol', 'validation_fraction', 'verbose', 'warm_start'].

# Classification Results

In [42]:
clf_df = pd.DataFrame(classification_results)
clf_df = clf_df.sort_values(by = 'F1',ascending=False)
clf_df.to_csv('Hyper results/class_results_hyper.csv',index=False)
clf_df

,Model,Accuracy,Precision,Recall,F1,ROC_AUC,Best_Params
10,Random Forest,0.768630,0.543796,0.796791,0.646421,0.856351,"{'model__max_depth': 10, 'model__max_features'..."
11,Random Forest,0.763662,0.538033,0.775401,0.635268,0.853813,"{'model__max_depth': 13, 'model__max_features'..."
9,Random Forest,0.760823,0.533945,0.778075,0.633297,0.852359,"{'model__max_depth': 15, 'model__max_features'..."
16,SVM,0.757275,0.528674,0.788770,0.633047,0.836658,"{'model__C': 1.0, 'model__gamma': 'scale', 'mo..."
2,SVM,0.757275,0.528674,0.788770,0.633047,0.836658,"{'model__C': 1.0, 'model__gamma': 'scale', 'mo..."
5,Random Forest,0.765791,0.543137,0.740642,0.626697,0.851851,"{'model__max_depth': 10, 'model__max_features'..."
15,Decision Tree,0.746629,0.515044,0.778075,0.619808,0.835900,"{'model__max_depth': 6, 'model__min_samples_le..."
13,Decision Tree,0.746629,0.515044,0.778075,0.619808,0.835900,"{'model__max_depth': 6, 'model__min_samples_le..."
14,Decision Tree,0.746629,0.515044,0.778075,0.619808,0.835900,"{'model__max_depth': 6, 'model__min_samples_le..."
0,Logistic Regression,0.743790,0.511344,0.783422,0.618796,0.848066,{'model__C': 12.5}
